In [3]:
"""
Soccer Twos — 3D  (pygame only, no OpenGL)
===========================================
2v2 Multi-Agent Soccer with a 3D perspective renderer.
Uses a hand-rolled painter's-algorithm 3D → 2D projection.

Dependency (ONE package only):
    pip install pygame

Controls:
    Mouse drag (left btn) — orbit camera
    Scroll wheel          — zoom in / out
    SPACE                 — pause / resume
    R                     — new game
    ESC                   — quit
"""

import math, random, sys
from dataclasses import dataclass
from typing import List, Optional, Tuple

try:
    import pygame
    from pygame.locals import *
except ImportError:
    sys.exit("pygame not installed.  Run:  pip install pygame")

# ═══════════════════════════════════════════════════════════════════════════
# Constants
# ═══════════════════════════════════════════════════════════════════════════
FIELD_W, FIELD_H = 500, 300
CX, CY = FIELD_W / 2, FIELD_H / 2
GOAL_H   = 90
GOAL_D   = 28          # goal depth (into field)
AGENT_R  = 9.0
BALL_R   = 7.5
MAX_STEP = 1200
STEPS_PER_FRAME = 1    # 1 = natural pace

WIN_W, WIN_H = 1280, 720
FPS = 60

GY = CY - GOAL_H / 2   # goal top-y in field coords

# Colours
C_ALPHA   = (55,  245, 135)
C_BETA    = (245,  55,  55)
C_WHITE   = (255, 255, 255)
C_BALL    = (238, 238, 238)
C_GRASS_A = ( 28, 105,  28)
C_GRASS_B = ( 33,  86,  33)
C_SKY     = ( 11,  28,  11)

goal_a = dict(x=0,              y=GY, w=GOAL_D, h=GOAL_H)
goal_b = dict(x=FIELD_W-GOAL_D, y=GY, w=GOAL_D, h=GOAL_H)

# ═══════════════════════════════════════════════════════════════════════════
# Camera / projection
# ═══════════════════════════════════════════════════════════════════════════
class Camera:
    def __init__(self):
        self.theta = math.radians(215)   # horizontal orbit
        self.phi   = math.radians(28)    # elevation
        self.r     = 580.0
        # camera looks at this world point
        self.tx, self.ty, self.tz = CX, CY, 20.0
        self._drag = None

    def _basis(self):
        """Return (eye, fwd, right, up2) unit vectors."""
        t, p = self.theta, self.phi
        ex = self.tx + self.r * math.cos(p) * math.sin(t)
        ey = self.ty + self.r * math.cos(p) * math.cos(t)
        ez = self.tz + self.r * math.sin(p)

        # forward
        fx = self.tx - ex; fy = self.ty - ey; fz = self.tz - ez
        fl = math.sqrt(fx*fx+fy*fy+fz*fz)
        fx/=fl; fy/=fl; fz/=fl

        # right = fwd × world_up(0,0,1)
        rx =  fy;  ry = -fx;  rz = 0.0
        rl = math.sqrt(rx*rx+ry*ry+rz*rz)
        if rl<1e-9: rx,ry,rz=1,0,0
        else: rx/=rl; ry/=rl; rz/=rl

        # camera up = right × fwd
        ux = ry*fz - rz*fy
        uy = rz*fx - rx*fz
        uz = rx*fy - ry*fx

        return (ex,ey,ez), (fx,fy,fz), (rx,ry,rz), (ux,uy,uz)

    def project(self, wx, wy, wz) -> Optional[Tuple[int,int]]:
        (ex,ey,ez),(fx,fy,fz),(rx,ry,rz),(ux,uy,uz) = self._basis()
        vx=wx-ex; vy=wy-ey; vz=wz-ez
        zc = vx*fx+vy*fy+vz*fz
        if zc < 1: return None
        xc = vx*rx+vy*ry+vz*rz
        yc = vx*ux+vy*uy+vz*uz
        scale = (WIN_H/2) / math.tan(math.radians(52)/2)
        sx = int(WIN_W/2 + xc/zc*scale)
        sy = int(WIN_H/2 - yc/zc*scale)
        return sx, sy

    def depth(self, wx, wy, wz) -> float:
        (ex,ey,ez),*_ = self._basis()
        dx=wx-ex; dy=wy-ey; dz=wz-ez
        return math.sqrt(dx*dx+dy*dy+dz*dz)

    def screen_radius(self, wx, wy, wz, world_r) -> int:
        """Approximate projected radius of a sphere."""
        p0 = self.project(wx, wy, wz)
        p1 = self.project(wx+world_r, wy, wz)
        if p0 and p1:
            return max(2, abs(p1[0]-p0[0]))
        return max(2, int(world_r))

    def mouse_down(self, pos):
        self._drag = (pos[0], pos[1], self.theta, self.phi)

    def mouse_move(self, pos):
        if not self._drag: return
        dx = pos[0]-self._drag[0]; dy = pos[1]-self._drag[1]
        self.theta = self._drag[2] + dx*0.005
        self.phi   = max(math.radians(8),
                     min(math.radians(72), self._drag[3] - dy*0.004))

    def mouse_up(self): self._drag = None
    def zoom(self, d):  self.r = max(280, min(1000, self.r+d))


cam = Camera()

# ═══════════════════════════════════════════════════════════════════════════
# Draw utilities
# ═══════════════════════════════════════════════════════════════════════════
def draw_line3(surf, p0, p1, col, w=2):
    a = cam.project(*p0); b = cam.project(*p1)
    if a and b: pygame.draw.line(surf, col[:3], a, b, w)

def draw_poly3(surf, pts3d, col, outline_col=None, outline_w=1):
    pts2 = [cam.project(*p) for p in pts3d]
    if any(p is None for p in pts2): return
    pygame.draw.polygon(surf, col[:3], pts2)
    if outline_col:
        pygame.draw.polygon(surf, outline_col[:3], pts2, outline_w)

def shade(col, factor):
    return tuple(max(0, min(255, int(c*factor))) for c in col[:3])

# ═══════════════════════════════════════════════════════════════════════════
# Field
# ═══════════════════════════════════════════════════════════════════════════
def draw_field(surf):
    # Depth-sort stripe quads
    sw = FIELD_W / 10
    stripes = []
    for i in range(10):
        col = C_GRASS_A if i%2==0 else C_GRASS_B
        x0=i*sw; x1=x0+sw
        d = cam.depth(x0+sw/2, CY, 0)
        stripes.append((d, x0, x1, col))
    stripes.sort(key=lambda s: -s[0])
    for _, x0, x1, col in stripes:
        draw_poly3(surf, [(x0,0,0),(x1,0,0),(x1,FIELD_H,0),(x0,FIELD_H,0)], col)

    # Boundary
    for a,b in [((0,0,0.2),(FIELD_W,0,0.2)),
                ((FIELD_W,0,0.2),(FIELD_W,FIELD_H,0.2)),
                ((FIELD_W,FIELD_H,0.2),(0,FIELD_H,0.2)),
                ((0,FIELD_H,0.2),(0,0,0.2))]:
        draw_line3(surf, a, b, C_WHITE, 2)

    # Centre line
    draw_line3(surf, (CX,0,0.2), (CX,FIELD_H,0.2), C_WHITE, 2)

    # Centre circle
    for i in range(40):
        a0=i/40*math.pi*2; a1=(i+1)/40*math.pi*2
        draw_line3(surf,
            (CX+math.cos(a0)*42, CY+math.sin(a0)*42, 0.2),
            (CX+math.cos(a1)*42, CY+math.sin(a1)*42, 0.2),
            (200,220,200), 1)

    # Penalty boxes
    def pbox(ox, oy, pw, ph):
        pts=[(ox,oy,0.2),(ox+pw,oy,0.2),(ox+pw,oy+ph,0.2),(ox,oy+ph,0.2)]
        draw_poly3(surf, pts, (0,0,0,0), outline_col=(200,220,200), outline_w=1)
    pbox(0, CY-65, 70, 130)
    pbox(FIELD_W-70, CY-65, 70, 130)


# ═══════════════════════════════════════════════════════════════════════════
# Goal
# ═══════════════════════════════════════════════════════════════════════════
def draw_goal(surf, side_x, team_col, facing):
    back_x = side_x + facing * GOAL_D
    y0, y1 = GY, GY + GOAL_H
    H = GOAL_H
    post_col   = C_WHITE
    post_dark  = shade(C_WHITE, 0.55)

    # Build render list: (depth, type, data)
    items = []

    # Net panels
    def net(pts, alpha=50):
        mx = sum(p[0] for p in pts)/len(pts)
        my = sum(p[1] for p in pts)/len(pts)
        mz = sum(p[2] for p in pts)/len(pts)
        items.append((cam.depth(mx,my,mz), 'net', pts, team_col, alpha))

    net([(back_x,y0,0),(back_x,y1,0),(back_x,y1,H),(back_x,y0,H)])
    net([(side_x,y0,H),(back_x,y0,H),(back_x,y1,H),(side_x,y1,H)])
    net([(side_x,y0,0),(back_x,y0,0),(back_x,y0,H),(side_x,y0,H)])
    net([(side_x,y1,0),(back_x,y1,0),(back_x,y1,H),(side_x,y1,H)])

    # Posts: thick 3-line stack (shadow + main + highlight)
    def post(x, y):
        d = cam.depth(x, y, H/2)
        items.append((d, 'post', (x,y,0,x,y,H), post_col))

    post(side_x, y0); post(side_x, y1)
    post(back_x, y0); post(back_x, y1)

    # Horizontal rods
    def rod(x0,y0_,z0, x1,y1_,z1, w=4):
        mx=(x0+x1)/2; my=(y0_+y1_)/2; mz=(z0+z1)/2
        items.append((cam.depth(mx,my,mz), 'rod', (x0,y0_,z0,x1,y1_,z1), post_col, w))

    # Front & back crossbars
    rod(side_x,y0,H, side_x,y1,H)
    rod(back_x,y0,H, back_x,y1,H)
    # Top depth rods
    rod(side_x,y0,H, back_x,y0,H)
    rod(side_x,y1,H, back_x,y1,H)
    # Ground depth rods
    rod(side_x,y0,0.5, back_x,y0,0.5, 3)
    rod(side_x,y1,0.5, back_x,y1,0.5, 3)

    # Goal mouth ring (coloured oval)
    segs=24
    for i in range(segs):
        a0=i/segs*math.pi*2; a1=(i+1)/segs*math.pi*2
        p0=(side_x, CY+math.cos(a0)*GOAL_H/2, H/2+math.sin(a0)*H/2)
        p1=(side_x, CY+math.cos(a1)*GOAL_H/2, H/2+math.sin(a1)*H/2)
        items.append((cam.depth(side_x,CY,H/2), 'rod', (*p0,*p1), team_col, 3))

    # Sort back → front
    items.sort(key=lambda e: -e[0])

    for el in items:
        kind = el[1]
        if kind == 'net':
            pts, col, alpha = el[2], el[3], el[4]
            p2 = [cam.project(*p) for p in pts]
            if any(p is None for p in p2): continue
            s = pygame.Surface((WIN_W,WIN_H), pygame.SRCALPHA)
            pygame.draw.polygon(s, (*col[:3], alpha), p2)
            surf.blit(s,(0,0))

        elif kind == 'post':
            x0,y0_,z0,x1,y1_,z1 = el[2]; col=el[3]
            pa=cam.project(x0,y0_,z0); pb=cam.project(x1,y1_,z1)
            if pa and pb:
                pygame.draw.line(surf, (20,20,20),    pa, pb, 8)
                pygame.draw.line(surf, shade(col,.6), pa, pb, 6)
                pygame.draw.line(surf, col,            pa, pb, 3)
                pygame.draw.line(surf, (230,255,230),  pa, pb, 1)

        elif kind == 'rod':
            x0,y0_,z0,x1,y1_,z1 = el[2]; col=el[3]; w=el[4]
            pa=cam.project(x0,y0_,z0); pb=cam.project(x1,y1_,z1)
            if pa and pb:
                pygame.draw.line(surf, (20,20,20), pa, pb, w+3)
                pygame.draw.line(surf, col,        pa, pb, w)


# ═══════════════════════════════════════════════════════════════════════════
# Agent
# ═══════════════════════════════════════════════════════════════════════════
def draw_agent(surf, a):
    col  = C_ALPHA if a.team==0 else C_BETA
    dark = shade(col, 0.35)
    x, y = a.x, a.y

    # Ground shadow
    sp = cam.project(x, y, 0.1)
    if sp:
        s = pygame.Surface((WIN_W,WIN_H), pygame.SRCALPHA)
        pygame.draw.ellipse(s, (0,0,0,70), (sp[0]-14,sp[1]-5,28,10))
        surf.blit(s,(0,0))

    # Body: draw as a set of horizontal rings stacked up (cylinder approx)
    body_h = 18; body_r = AGENT_R-1
    rings = 4
    for ri in range(rings):
        frac = ri/(rings-1)
        z = frac * body_h
        ring_r = body_r * (1 - 0.15*frac)    # slight taper
        bright = 0.5 + 0.5*(1-frac)
        c = shade(col, bright)

        # draw as a small filled circle at the projected position
        p = cam.project(x, y, z)
        if p:
            sr = cam.screen_radius(x, y, z, ring_r)
            pygame.draw.circle(surf, c, p, sr)

    # Solid flat disc for top face
    p_top = cam.project(x, y, body_h)
    if p_top:
        sr = cam.screen_radius(x, y, body_h, body_r)
        pygame.draw.circle(surf, col, p_top, sr)
        pygame.draw.circle(surf, shade(col, 1.3), p_top, max(1,sr-3))

    # Head
    p_head = cam.project(x, y, body_h+8)
    if p_head:
        sr = cam.screen_radius(x, y, body_h+8, 6)
        pygame.draw.circle(surf, dark, p_head, sr)
        pygame.draw.circle(surf, shade(dark,1.6), p_head, max(1,sr//3))

    # Direction arrow
    ax = x + math.cos(a.angle)*(body_r+10)
    ay = y + math.sin(a.angle)*(body_r+10)
    pa = cam.project(x, y, body_h/2)
    pb = cam.project(ax, ay, body_h/2)
    if pa and pb:
        pygame.draw.line(surf, C_WHITE, pa, pb, 3)
        pygame.draw.circle(surf, C_WHITE, pb, 4)

    # State dot
    sc = {"attack":(255,255,50),"support":(60,160,255)}.get(a.state,(180,180,180))
    p_dot = cam.project(x, y, body_h+18)
    if p_dot:
        pygame.draw.circle(surf, sc, p_dot, 4)


# ═══════════════════════════════════════════════════════════════════════════
# Ball
# ═══════════════════════════════════════════════════════════════════════════
def draw_ball(surf, b):
    x, y = b.x, b.y

    # Shadow
    sp = cam.project(x, y, 0.2)
    if sp:
        s = pygame.Surface((WIN_W,WIN_H), pygame.SRCALPHA)
        pygame.draw.ellipse(s,(0,0,0,65),(sp[0]-10,sp[1]-4,20,8))
        surf.blit(s,(0,0))

    cz = b.r + 2
    c  = cam.project(x, y, cz)
    if not c: return
    sr = cam.screen_radius(x, y, cz, b.r)

    # Body
    pygame.draw.circle(surf, C_BALL, c, sr)
    pygame.draw.circle(surf, (130,130,130), c, sr, 1)

    # Spinning dark pentagon panels
    for i in range(5):
        ang = i/5*math.pi*2 + b.spin
        px = x + math.cos(ang)*b.r*0.55
        py = y + math.sin(ang)*b.r*0.55
        pp = cam.project(px, py, cz)
        if pp:
            pygame.draw.circle(surf, (35,35,35), pp, max(2,int(sr*0.27)))

    # Highlight
    hp = cam.project(x - b.r*0.3, y - b.r*0.3, cz + b.r*0.5)
    if hp:
        pygame.draw.circle(surf, (255,255,255), hp, max(1,int(sr*0.18)))


# ═══════════════════════════════════════════════════════════════════════════
# Simulation logic
# ═══════════════════════════════════════════════════════════════════════════
@dataclass
class Ball:
    x: float = CX; y: float = CY
    vx: float = 0.0; vy: float = 0.0
    r: float = BALL_R; spin: float = 0.0

@dataclass
class Agent:
    x: float = 0.0; y: float = 0.0
    vx: float = 0.0; vy: float = 0.0
    angle: float = 0.0; team: int = 0
    role: str = "striker"; speed: float = 2.4
    state: str = "chase"

def make_agents():
    return [
        Agent(x=CX-100,y=CY-25,angle=0,       team=0,role="striker", speed=2.4),
        Agent(x=CX-80, y=CY+25,angle=0,       team=0,role="defender",speed=2.1),
        Agent(x=CX+100,y=CY-25,angle=math.pi, team=1,role="striker", speed=2.4),
        Agent(x=CX+80, y=CY+25,angle=math.pi, team=1,role="defender",speed=2.1),
    ]

def reset_positions(ball, agents):
    ball.x,ball.y=CX,CY
    ball.vx=random.uniform(-3,3); ball.vy=random.uniform(-2.5,2.5)
    for a,(sx,sy) in zip(agents,[(CX-100,CY-25),(CX-80,CY+25),(CX+100,CY-25),(CX+80,CY+25)]):
        a.x,a.y=sx,sy

def agent_ai(a,idx,agents,ball):
    if a.team==0:
        og=(goal_b["x"]+goal_b["w"]/2,goal_b["y"]+goal_b["h"]/2)
        own=(goal_a["x"]+goal_a["w"]/2,goal_a["y"]+goal_a["h"]/2)
    else:
        og=(goal_a["x"]+goal_a["w"]/2,goal_a["y"]+goal_a["h"]/2)
        own=(goal_b["x"]+goal_b["w"]/2,goal_b["y"]+goal_b["h"]/2)
    tms=[ag for i,ag in enumerate(agents) if ag.team==a.team and i!=idx]
    db=math.hypot(a.x-ball.x,a.y-ball.y)
    nearest=(not tms) or db<min(math.hypot(t.x-ball.x,t.y-ball.y) for t in tms)
    if nearest or a.role=="striker":
        if db>18: tx,ty=ball.x,ball.y; a.state="chase"
        else:
            dx,dy=og[0]-a.x,og[1]-a.y; nd=math.hypot(dx,dy) or 1
            tx,ty=ball.x+dx/nd*80,ball.y+dy/nd*80; a.state="attack"
    else:
        tx,ty=ball.x*0.4+own[0]*0.6,ball.y*0.5+own[1]*0.5; a.state="support"
    err=math.atan2(ty-a.y,tx-a.x)-a.angle
    while err>math.pi: err-=2*math.pi
    while err<-math.pi: err+=2*math.pi
    rot=2 if err>0.10 else 1 if err<-0.10 else 0
    fwd=1 if abs(err)<math.pi*.5 else 2 if abs(err)>math.pi*.85 else 1
    return fwd,rot

def apply_action(a,fwd,rot):
    TR=0.15
    if rot==2: a.angle+=TR
    elif rot==1: a.angle-=TR
    f=a.speed*(1.0 if fwd==1 else -0.55 if fwd==2 else 0.0)
    a.vx=math.cos(a.angle)*f; a.vy=math.sin(a.angle)*f
    a.x=max(AGENT_R,min(FIELD_W-AGENT_R,a.x+a.vx))
    a.y=max(AGENT_R,min(FIELD_H-AGENT_R,a.y+a.vy))

def update_ball(b):
    b.vx*=0.994; b.vy*=0.994; b.x+=b.vx; b.y+=b.vy
    b.spin+=math.hypot(b.vx,b.vy)*0.04
    in_a=goal_a["y"]<=b.y<=goal_a["y"]+goal_a["h"]
    in_b=goal_b["y"]<=b.y<=goal_b["y"]+goal_b["h"]
    if b.x-b.r<0 and not in_a:        b.x=b.r;          b.vx= abs(b.vx)*.75
    if b.x+b.r>FIELD_W and not in_b:  b.x=FIELD_W-b.r;  b.vx=-abs(b.vx)*.75
    if b.y-b.r<0:                      b.y=b.r;           b.vy= abs(b.vy)*.75
    if b.y+b.r>FIELD_H:               b.y=FIELD_H-b.r;   b.vy=-abs(b.vy)*.75

def agent_ball_col(a,b):
    d=math.hypot(a.x-b.x,a.y-b.y); mn=AGENT_R+b.r
    if d<mn and d>0.01:
        nx=(b.x-a.x)/d; ny=(b.y-a.y)/d
        rv=(b.vx-a.vx)*nx+(b.vy-a.vy)*ny
        if rv<0:
            imp=-rv*2.5; b.vx+=nx*imp; b.vy+=ny*imp
            og=goal_b if a.team==0 else goal_a
            gcx=og["x"]+og["w"]/2; gcy=og["y"]+og["h"]/2
            bd=math.hypot(gcx-b.x,gcy-b.y) or 1
            b.vx+=(gcx-b.x)/bd*2.2; b.vy+=(gcy-b.y)/bd*2.2
        ov=mn-d+0.5
        b.x+=nx*ov*.6; b.y+=ny*ov*.6; a.x-=nx*ov*.4; a.y-=ny*ov*.4

def agent_sep(agents):
    for i in range(len(agents)):
        for j in range(i+1,len(agents)):
            a,b=agents[i],agents[j]; d=math.hypot(a.x-b.x,a.y-b.y); mn=AGENT_R*2
            if d<mn and d>0.01:
                nx=(a.x-b.x)/d; ny=(a.y-b.y)/d; p=(mn-d)/2
                a.x+=nx*p; a.y+=ny*p; b.x-=nx*p; b.y-=ny*p

def check_goal(b):
    if b.x-b.r<=GOAL_D and goal_a["y"]-4<=b.y<=goal_a["y"]+GOAL_H+4: return 1
    if b.x+b.r>=FIELD_W-GOAL_D and goal_b["y"]-4<=b.y<=goal_b["y"]+GOAL_H+4: return 0
    return -1


# ═══════════════════════════════════════════════════════════════════════════
# HUD
# ═══════════════════════════════════════════════════════════════════════════
def draw_hud(surf, fb, fs, gs, goal_flash, goal_team):
    sc=gs["score"]
    t=fb.render(f"ALPHA  {sc[0]}  :  {sc[1]}  BETA",True,(195,255,205))
    surf.blit(t,(WIN_W//2-t.get_width()//2,10))

    info=fs.render(f"Ep {gs['ep']}   Step {gs['step']}/{MAX_STEP}   "
                   f"Wins  A:{gs['wins'][0]}  B:{gs['wins'][1]}  D:{gs['draws']}",
                   True,(145,195,145))
    surf.blit(info,(WIN_W//2-info.get_width()//2,42))

    if gs.get("paused"):
        tp=fb.render("PAUSED",True,(255,215,50))
        surf.blit(tp,(WIN_W//2-tp.get_width()//2,WIN_H//2-20))

    if goal_flash>0:
        alpha=int(math.sin(goal_flash/70*math.pi)*190)
        fc=C_ALPHA if goal_team==0 else C_BETA
        fs2=pygame.Surface((WIN_W,WIN_H),pygame.SRCALPHA)
        fs2.fill((*fc,alpha//5)); surf.blit(fs2,(0,0))
        tg=fb.render("GOAL!",True,fc)
        surf.blit(tg,(WIN_W//2-tg.get_width()//2,WIN_H//2-tg.get_height()//2))

    for i,line in enumerate(gs["log"][-5:]):
        tl=fs.render(line,True,(155,205,135))
        surf.blit(tl,(12,WIN_H-22-(4-i)*18))

    hint=fs.render("Drag: orbit   Scroll: zoom   SPACE: pause   R: new game   ESC: quit",
                   True,(65,105,65))
    surf.blit(hint,(WIN_W//2-hint.get_width()//2,WIN_H-18))


# ═══════════════════════════════════════════════════════════════════════════
# Main loop
# ═══════════════════════════════════════════════════════════════════════════
def new_game():
    return dict(step=0,score=[0,0],ep=1,
                ball=Ball(vx=random.uniform(-3,3),vy=random.uniform(-2.5,2.5)),
                agents=make_agents(),wins=[0,0],draws=0,total_goals=0,log=[],paused=False)

def main():
    pygame.init()
    screen=pygame.display.set_mode((WIN_W,WIN_H))
    pygame.display.set_caption("Soccer Twos — 3D  |  2v2 Multi-Agent")
    clock=pygame.time.Clock()
    fb=pygame.font.SysFont("monospace",22,bold=True)
    fs=pygame.font.SysFont("monospace",13)

    gs=new_game()
    paused=False; goal_flash=0; goal_team=-1

    def logi(msg):
        gs["log"].append(msg)
        if len(gs["log"])>8: gs["log"].pop(0)

    running=True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type==QUIT: running=False
            elif event.type==KEYDOWN:
                if event.key==K_ESCAPE: running=False
                elif event.key==K_SPACE: paused=not paused; gs["paused"]=paused
                elif event.key==K_r: gs=new_game(); goal_flash=0
            elif event.type==MOUSEBUTTONDOWN:
                if event.button==1: cam.mouse_down(event.pos)
                elif event.button==4: cam.zoom(-50)
                elif event.button==5: cam.zoom(50)
            elif event.type==MOUSEBUTTONUP:
                if event.button==1: cam.mouse_up()
            elif event.type==MOUSEMOTION:
                cam.mouse_move(event.pos)

        if not paused:
            for _ in range(STEPS_PER_FRAME):
                ball=gs["ball"]; agents=gs["agents"]
                for i,a in enumerate(agents):
                    fwd,rot=agent_ai(a,i,agents,ball); apply_action(a,fwd,rot)
                update_ball(ball)
                for a in agents: agent_ball_col(a,ball)
                agent_sep(agents)
                scored=check_goal(ball)
                if scored>=0:
                    gs["score"][scored]+=1; gs["total_goals"]+=1
                    goal_flash=70; goal_team=scored
                    nm="ALPHA" if scored==0 else "BETA"
                    logi(f"[Ep{gs['ep']:02d} S{gs['step']:04d}] GOAL! {nm} → {gs['score'][0]}-{gs['score'][1]}")
                    reset_positions(ball,agents)
                gs["step"]+=1
                if gs["step"]>=MAX_STEP:
                    a_s,b_s=gs["score"]
                    if a_s>b_s: gs["wins"][0]+=1; res="ALPHA wins"
                    elif b_s>a_s: gs["wins"][1]+=1; res="BETA wins"
                    else: gs["draws"]+=1; res="Draw"
                    logi(f"Ep{gs['ep']:02d}: {a_s}-{b_s} | {res}")
                    gs["ep"]+=1; gs["step"]=0; gs["score"]=[0,0]
                    reset_positions(ball,agents)
            if goal_flash>0: goal_flash-=1

        # ── Render ──────────────────────────────────────────────────────────
        screen.fill(C_SKY)
        draw_field(screen)

        # Goals: draw farther one first
        ga_d=cam.depth(0,      CY,GOAL_H/2)
        gb_d=cam.depth(FIELD_W,CY,GOAL_H/2)
        if ga_d>gb_d:
            draw_goal(screen,0,      C_ALPHA,+1)
            draw_goal(screen,FIELD_W,C_BETA, -1)
        else:
            draw_goal(screen,FIELD_W,C_BETA, -1)
            draw_goal(screen,0,      C_ALPHA,+1)

        # Agents + ball, sorted back-to-front
        ball=gs["ball"]; agents=gs["agents"]
        items=[(cam.depth(a.x,a.y,12),'a',a) for a in agents]
        items.append((cam.depth(ball.x,ball.y,ball.r+2),'b',ball))
        items.sort(key=lambda e:-e[0])
        for _,kind,obj in items:
            if kind=='a': draw_agent(screen,obj)
            else:         draw_ball(screen,obj)

        draw_hud(screen,fb,fs,gs,goal_flash,goal_team)
        pygame.display.flip()

    pygame.quit()

if __name__=="__main__":
    main()